# Naver 중심 종이신문 이미지 수집 — 최종 6,000장 목표

Google Selenium은 사용하지 않습니다. **네이버 이미지 검색을 중심**으로 실제 종이신문/신문지/폐신문 사진을 수집합니다.

- 무료, 별도 API Key 불필요
- Selenium + Chrome
- 한국어 종이신문 검색어 중심
- 검색어별 최대 300장
- 0장 수집 시 검색어 폴더 생성 안 함
- 기존 이미지가 있으면 이어서 수집
- 브라우저 세션 종료 시 자동 재시작
- 실제 `image/*` + Pillow 검증 후 저장
- 저해상도 제거
- SHA-256 완전중복 제거
- pHash 유사후보 CSV
- pandas 미사용 / 상대경로

최종 6,000장을 남기려면 RAW는 약 10,000~15,000장을 목표로 합니다. 검색 결과 및 사이트 정책에 따라 실제 수집량은 달라질 수 있습니다.


## 1. Import

In [1]:
from pathlib import Path
from io import BytesIO
from datetime import datetime
import csv, hashlib, shutil, time, urllib.parse, urllib.request, random
from PIL import Image, ImageOps
import imagehash
from tqdm.auto import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    InvalidSessionIdException, NoSuchWindowException,
    WebDriverException, StaleElementReferenceException
)
print("Import 완료")


Import 완료


## 2. 설정

In [2]:
KEYWORDS = [
    "실물 종이신문","종이신문 사진","종이 신문 펼쳐진","종이 신문 접힌",
    "종이신문 책상","종이신문 바닥","종이신문 읽는 모습","손에 든 종이신문",
    "종이신문 더미","종이신문 묶음","종이신문 여러 장",
    "신문지 펼쳐진","신문지 접힌","신문지 구겨진","신문지 바닥","신문지 책상",
    "신문지 더미","신문지 묶음","신문지 쌓아놓은","신문지 재활용","신문지 분리수거",
    "폐신문","폐신문 사진","폐신문 더미","폐신문 묶음",
    "폐신문지","폐신문지 더미","폐신문지 묶음",
    "헌신문","헌신문 더미","헌신문 묶음",
    "신문 폐지 수거","폐지 신문지","신문지 재활용품","신문지 수거",
    "신문 폐지 더미","폐신문 재활용","신문지 폐지","헌 신문지",
    "신문 재활용","신문지 수거함","폐지 수거 신문","신문 묶어놓은 사진",
    "신문 쌓아놓은 사진","종이신문 재활용","신문지 쓰레기","신문 폐기",
    "신문지 수거 사진","종이신문 폐기","신문지 분리배출"
]

TARGET_PER_KEYWORD = 300
MAX_SCROLLS = 60
NO_GROWTH_LIMIT = 8
SCROLL_WAIT = 1.2
CLICK_WAIT = 0.25
REQUEST_TIMEOUT = 12
MIN_WIDTH, MIN_HEIGHT = 300, 300
MAX_SESSION_RESTARTS = 8
PHASH_DISTANCE_THRESHOLD = 4
HEADLESS = False

BASE_DIR = Path.cwd() / "naver_newspaper_collection_6000"
RAW_DIR = BASE_DIR / "01_raw_images"
VALID_DIR = BASE_DIR / "02_valid_images"
DEDUP_DIR = BASE_DIR / "03_deduplicated_images"
REVIEW_DIR = BASE_DIR / "04_review_images"
REPORT_DIR = BASE_DIR / "reports"

for d in [RAW_DIR, VALID_DIR, DEDUP_DIR, REVIEW_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("저장 위치:", BASE_DIR.resolve())
print("검색어 수:", len(KEYWORDS))
print("이론상 최대:", len(KEYWORDS) * TARGET_PER_KEYWORD)


저장 위치: C:\Users\mingy\OneDrive\Desktop\새 폴더\naver_newspaper_collection_6000
검색어 수: 50
이론상 최대: 15000


## 3. 공통 함수

In [ ]:
EXTS={".jpg",".jpeg",".png",".webp",".bmp",".gif"}
FMT={"JPEG":".jpg","PNG":".png","WEBP":".webp","BMP":".bmp","GIF":".gif"}
MANIFEST=REPORT_DIR/"download_manifest.csv"
PROGRESS=REPORT_DIR/"progress.csv"

def safe_name(s):
    for c in '<>:"/\\|?*': s=s.replace(c,"_")
    return s.strip()

def images(root):
    root=Path(root)
    if not root.exists(): return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)

def append_csv(path,row,fields):
    new=not path.exists()
    with path.open("a",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=fields)
        if new: w.writeheader()
        w.writerow(row)

def write_csv(path,rows,fields):
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def robust_rmtree(path, retries=8, delay=1.0):
    # OneDrive 동기화가 폴더를 일시적으로 잠그면 rmtree가 WinError 5로 실패하므로 재시도
    path=Path(path)
    if not path.exists(): return
    for attempt in range(retries):
        try:
            shutil.rmtree(path)
            return
        except (PermissionError, OSError):
            if attempt==retries-1: raise
            time.sleep(delay*(attempt+1))

def make_driver():
    o=Options()
    o.add_argument("--lang=ko-KR")
    o.add_argument("--window-size=1450,1000")
    o.add_argument("--disable-notifications")
    o.add_argument("--disable-popup-blocking")
    if HEADLESS: o.add_argument("--headless=new")
    return webdriver.Chrome(options=o)

def alive(d):
    try: _=d.current_url; return True
    except: return False

def restart(d=None):
    try:
        if d: d.quit()
    except: pass
    print("  Chrome 새 세션 시작")
    return make_driver()

def download(url):
    if not url or not url.startswith(("http://","https://")):
        raise ValueError("bad_url")
    req=urllib.request.Request(url,headers={
        "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/153 Safari/537.36",
        "Accept":"image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Referer":"https://search.naver.com/"
    })
    with urllib.request.urlopen(req,timeout=REQUEST_TIMEOUT) as r:
        ct=r.headers.get("Content-Type","")
        data=r.read(25*1024*1024+1)
    if not ct.lower().startswith("image/"): raise ValueError("not_image")
    if len(data)>25*1024*1024: raise ValueError("too_large")
    with Image.open(BytesIO(data)) as im: im.verify()
    with Image.open(BytesIO(data)) as im:
        fmt=(im.format or "").upper(); w,h=im.size
    ext=FMT.get(fmt)
    if not ext: raise ValueError("unsupported")
    return data,ext,w,h,ct

def candidate_urls(driver):
    # Naver DOM class 이름 대신 현재 문서의 img src/currentSrc/data-*를 폭넓게 사용
    urls=driver.execute_script("""
    const out=[];
    for(const img of document.images){
      const attrs=[
        img.currentSrc, img.src,
        img.getAttribute('data-src'),
        img.getAttribute('data-lazy-src'),
        img.getAttribute('data-original')
      ];
      for(const s of attrs){
        if(!s || !s.startsWith('http')) continue;
        const w=img.naturalWidth||0, h=img.naturalHeight||0;
        if(w>=200 && h>=150) out.push([s,w*h]);
      }
    }
    out.sort((a,b)=>b[1]-a[1]);
    return out.map(x=>x[0]);
    """)
    return list(dict.fromkeys(urls))

print("함수 준비 완료")


## 4. Naver 이미지 검색 테스트

In [4]:
driver=make_driver()
test_url="https://search.naver.com/search.naver?where=image&sm=tab_jum&query="+urllib.parse.quote_plus("실물 종이신문")
driver.get(test_url)
time.sleep(2)
print("페이지:",driver.title)
print("URL:",driver.current_url)
print("세션 정상:",alive(driver))


페이지: 실물 종이신문 : 네이버 이미지검색
URL: https://search.naver.com/search.naver?where=image&sm=tab_jum&query=%EC%8B%A4%EB%AC%BC+%EC%A2%85%EC%9D%B4%EC%8B%A0%EB%AC%B8
세션 정상: True


## 5. 메인 수집

검색어 폴더는 **첫 이미지 저장에 성공한 순간에만 생성**됩니다. 기존 폴더에 이미지가 있으면 이어서 수집합니다.


In [5]:
MF=["time","keyword","filename","image_url","width","height","content_type","status"]
PF=["time","keyword","existing_before","images_now","target","scrolls","status"]
seen_urls=set()
restarts=0
stop=False

if "driver" not in globals() or not alive(driver):
    driver=restart()

for ki,kw in enumerate(KEYWORDS,1):
    if stop: break

    folder=RAW_DIR/f"{ki:02d}_{safe_name(kw)}"
    existing=len(images(folder)) if folder.exists() else 0
    saved=existing

    print("\n"+"="*76)
    print(f"[{ki}/{len(KEYWORDS)}] {kw} | 기존 {existing}장")
    print("="*76)

    if saved>=TARGET_PER_KEYWORD:
        print("SKIP: 이미 목표 수량 완료")
        continue

    search_url="https://search.naver.com/search.naver?where=image&sm=tab_jum&query="+urllib.parse.quote_plus(kw)

    try:
        if not alive(driver): driver=restart(driver)
        driver.get(search_url); time.sleep(2)
    except Exception:
        driver=restart(driver); driver.get(search_url); time.sleep(2)

    scrolls=0
    no_growth=0
    last=saved
    status="finished"

    while saved<TARGET_PER_KEYWORD and scrolls<MAX_SCROLLS:
        try:
            if not alive(driver):
                raise InvalidSessionIdException("session lost")

            # 현재 페이지 이미지 URL 직접 수집
            urls=candidate_urls(driver)

            # 보이는 썸네일 클릭으로 큰 이미지가 로드되는 경우도 확보
            elems=driver.find_elements(By.CSS_SELECTOR,"img")
            for el in elems[-100:]:
                if saved>=TARGET_PER_KEYWORD: break
                try:
                    sz=el.size
                    if sz.get("width",0)>=90 and sz.get("height",0)>=65:
                        driver.execute_script("arguments[0].scrollIntoView({block:'center'});",el)
                        el.click()
                        time.sleep(CLICK_WAIT)
                        urls.extend(candidate_urls(driver))
                except (StaleElementReferenceException,WebDriverException):
                    continue
                except Exception:
                    continue

            urls=list(dict.fromkeys(urls))

            for iu in urls:
                if saved>=TARGET_PER_KEYWORD: break
                if iu in seen_urls: continue
                seen_urls.add(iu)

                try:
                    raw,ext,w,h,ct=download(iu)
                    if w<MIN_WIDTH or h<MIN_HEIGHT:
                        continue

                    # 첫 성공 이미지가 생길 때만 폴더 생성
                    folder.mkdir(parents=True,exist_ok=True)
                    saved+=1
                    fn=f"{saved:05d}{ext}"
                    (folder/fn).write_bytes(raw)

                    append_csv(MANIFEST,{
                        "time":datetime.now().isoformat(timespec="seconds"),
                        "keyword":kw,"filename":fn,"image_url":iu,
                        "width":w,"height":h,"content_type":ct,"status":"downloaded"
                    },MF)
                except Exception as e:
                    append_csv(MANIFEST,{
                        "time":datetime.now().isoformat(timespec="seconds"),
                        "keyword":kw,"filename":"","image_url":iu,
                        "width":"","height":"","content_type":"",
                        "status":f"failed:{type(e).__name__}"
                    },MF)

                time.sleep(random.uniform(.08,.18))

            if saved==last:
                no_growth+=1
            else:
                no_growth=0
                last=saved

            print(f"\r저장 {saved}/{TARGET_PER_KEYWORD} | 스크롤 {scrolls+1}/{MAX_SCROLLS} | 무증가 {no_growth}/{NO_GROWTH_LIMIT}",end="")

            if no_growth>=NO_GROWTH_LIMIT:
                status="no_more_growth"
                break

            driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
            time.sleep(SCROLL_WAIT)
            scrolls+=1

        except (InvalidSessionIdException,NoSuchWindowException,WebDriverException):
            print("\nChrome 세션 끊김 → 자동 재시작")
            restarts+=1
            if restarts>MAX_SESSION_RESTARTS:
                status="restart_limit"; stop=True; break
            driver=restart(driver)
            driver.get(search_url)
            time.sleep(2)
            scrolls=0
            no_growth=0

    current=len(images(folder)) if folder.exists() else 0
    print(f"\n현재 저장: {current}장" if current else "\n0장 → 검색어 폴더 생성 안 됨")

    append_csv(PROGRESS,{
        "time":datetime.now().isoformat(timespec="seconds"),
        "keyword":kw,"existing_before":existing,"images_now":current,
        "target":TARGET_PER_KEYWORD,"scrolls":scrolls,
        "status":"target_reached" if current>=TARGET_PER_KEYWORD else status
    },PF)

print("\nRAW 총 이미지:",len(images(RAW_DIR)))
print("위치:",RAW_DIR.resolve())
print("세션 재시작:",restarts)



[1/50] 실물 종이신문 | 기존 300장
SKIP: 이미 목표 수량 완료

[2/50] 종이신문 사진 | 기존 293장
저장 300/300 | 스크롤 1/60 | 무증가 0/8
현재 저장: 300장

[3/50] 종이 신문 펼쳐진 | 기존 243장
저장 300/300 | 스크롤 1/60 | 무증가 0/8
현재 저장: 300장

[4/50] 종이 신문 접힌 | 기존 150장
저장 300/300 | 스크롤 2/60 | 무증가 0/8
현재 저장: 300장

[5/50] 종이신문 책상 | 기존 83장
저장 300/300 | 스크롤 3/60 | 무증가 0/8
현재 저장: 300장

[6/50] 종이신문 바닥 | 기존 71장
저장 300/300 | 스크롤 4/60 | 무증가 0/8
현재 저장: 300장

[7/50] 종이신문 읽는 모습 | 기존 300장
SKIP: 이미 목표 수량 완료

[8/50] 손에 든 종이신문 | 기존 210장
저장 300/300 | 스크롤 2/60 | 무증가 0/8
현재 저장: 300장

[9/50] 종이신문 더미 | 기존 179장
저장 300/300 | 스크롤 3/60 | 무증가 0/8
현재 저장: 300장

[10/50] 종이신문 묶음 | 기존 98장
저장 235/300 | 스크롤 12/60 | 무증가 8/8
현재 저장: 235장

[11/50] 종이신문 여러 장 | 기존 140장
저장 300/300 | 스크롤 3/60 | 무증가 0/8
현재 저장: 300장

[12/50] 신문지 펼쳐진 | 기존 300장
SKIP: 이미 목표 수량 완료

[13/50] 신문지 접힌 | 기존 300장
SKIP: 이미 목표 수량 완료

[14/50] 신문지 구겨진 | 기존 300장
SKIP: 이미 목표 수량 완료

[15/50] 신문지 바닥 | 기존 210장
저장 300/300 | 스크롤 2/60 | 무증가 0/8
현재 저장: 300장

[16/50] 신문지 책상 | 기존 0장
저장 300/300 | 스크롤 3/60 | 무증가 0/8
현재 저장: 300장


## 6. Chrome 종료

In [6]:
try:
    driver.quit()
except:
    pass
print("Chrome 종료")


Chrome 종료


## 7. 검색어별 수집량

In [7]:
total=0
for i,k in enumerate(KEYWORDS,1):
    f=RAW_DIR/f"{i:02d}_{safe_name(k)}"
    n=len(images(f)) if f.exists() else 0
    total+=n
    print(f"{i:02d}. {k:<25} {n:>4}장", "폴더 없음" if n==0 and not f.exists() else "")
print("-"*65)
print("RAW 총합:",total)


01. 실물 종이신문                    300장 
02. 종이신문 사진                    300장 
03. 종이 신문 펼쳐진                  300장 
04. 종이 신문 접힌                   300장 
05. 종이신문 책상                    300장 
06. 종이신문 바닥                    300장 
07. 종이신문 읽는 모습                 300장 
08. 손에 든 종이신문                  300장 
09. 종이신문 더미                    300장 
10. 종이신문 묶음                    235장 
11. 종이신문 여러 장                  300장 
12. 신문지 펼쳐진                    300장 
13. 신문지 접힌                     300장 
14. 신문지 구겨진                    300장 
15. 신문지 바닥                     300장 
16. 신문지 책상                     300장 
17. 신문지 더미                     300장 
18. 신문지 묶음                     300장 
19. 신문지 쌓아놓은                   300장 
20. 신문지 재활용                    300장 
21. 신문지 분리수거                   300장 
22. 폐신문                        300장 
23. 폐신문 사진                      47장 
24. 폐신문 더미                       3장 
25. 폐신문 묶음                      23장 
26. 폐신문지                       300장 
27. 폐신문지 더미                     13장 
2

## 8. 품질검사

In [ ]:
robust_rmtree(VALID_DIR)
VALID_DIR.mkdir(parents=True)
rows=[]
for src in tqdm(images(RAW_DIR),desc="품질검사"):
    try:
        with Image.open(src) as im:
            im=ImageOps.exif_transpose(im); w,h=im.size
        ok=w>=MIN_WIDTH and h>=MIN_HEIGHT
        if ok:
            dst=VALID_DIR/src.relative_to(RAW_DIR)
            dst.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(src,dst)
        rows.append({"file":str(src.relative_to(RAW_DIR)),"width":w,"height":h,"status":"valid" if ok else "too_small"})
    except:
        rows.append({"file":str(src.relative_to(RAW_DIR)),"width":"","height":"","status":"invalid"})
write_csv(REPORT_DIR/"validation.csv",rows,["file","width","height","status"])
print("RAW:",len(images(RAW_DIR)),"VALID:",len(images(VALID_DIR)))


## 9. SHA-256 완전중복 제거

In [ ]:
robust_rmtree(DEDUP_DIR)
DEDUP_DIR.mkdir(parents=True)
seen={}; dups=[]
for src in tqdm(images(VALID_DIR),desc="완전중복"):
    h=sha256(src); rel=src.relative_to(VALID_DIR)
    if h in seen:
        dups.append({"duplicate":str(rel),"kept":seen[h],"sha256":h})
        continue
    seen[h]=str(rel)
    dst=DEDUP_DIR/rel
    dst.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(src,dst)
write_csv(REPORT_DIR/"exact_duplicates.csv",dups,["duplicate","kept","sha256"])
print("완전중복:",len(dups))
print("중복 제거 후:",len(images(DEDUP_DIR)))


## 10. pHash 유사후보

In [10]:
rec=[]
for p in tqdm(images(DEDUP_DIR),desc="pHash"):
    try:
        with Image.open(p) as im:
            rec.append((p,imagehash.phash(im.convert("RGB"))))
    except:
        pass
b={}
for p,h in rec: b.setdefault(str(h)[:2],[]).append((p,h))
sim=[]
for items in tqdm(b.values(),desc="유사후보"):
    for i in range(len(items)):
        for j in range(i+1,len(items)):
            d=items[i][1]-items[j][1]
            if d<=PHASH_DISTANCE_THRESHOLD:
                sim.append({
                    "file_a":str(items[i][0].relative_to(DEDUP_DIR)),
                    "file_b":str(items[j][0].relative_to(DEDUP_DIR)),
                    "distance":d
                })
write_csv(REPORT_DIR/"similar_candidates.csv",sim,["file_a","file_b","distance"])
print("유사 후보 쌍:",len(sim))


pHash:   0%|          | 0/8159 [00:00<?, ?it/s]

유사후보:   0%|          | 0/128 [00:00<?, ?it/s]

유사 후보 쌍: 2246


## 11. 최종 육안검수 폴더

In [ ]:
robust_rmtree(REVIEW_DIR)
REVIEW_DIR.mkdir(parents=True)
for src in tqdm(images(DEDUP_DIR),desc="검수폴더"):
    dst=REVIEW_DIR/src.relative_to(DEDUP_DIR)
    dst.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(src,dst)

raw=len(images(RAW_DIR))
valid=len(images(VALID_DIR))
dedup=len(images(DEDUP_DIR))

print(f"RAW                 : {raw:,}")
print(f"품질검사 통과        : {valid:,}")
print(f"완전중복 제거 후      : {dedup:,}")
print(f"육안검수 대상         : {dedup:,}")
print(f"6,000장까지 최소      : {max(0,6000-dedup):,}장 (육안검수 전)")
print("검수 위치:",REVIEW_DIR.resolve())


## 육안검수 기준

**남기기**
- 실제 종이신문/신문지가 주요 객체
- 펼침, 접힘, 구김
- 폐신문 더미/묶음
- 바닥/책상에 놓인 신문
- 사람이 실제 종이신문을 들거나 읽는 사진

**제거**
- 인터넷 뉴스 캡처
- 기사에 첨부된 인물/사건 사진만 있는 이미지
- 방송 화면
- 로고/일러스트
- 잡지/책/전단지
- 신문이 너무 작게 나온 사진
- 동일 장면의 과도한 반복

검색 결과의 이미지에는 제3자 저작권이 있을 수 있으므로 학습 데이터의 공개·재배포 전에는 각 출처의 이용 조건을 별도로 확인하세요.
